# Persistent Foundry Agent

Creates a persistent agent on Azure AI Foundry, sends a user message, and streams the response back in real time.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

In [ ]:
# Create a persistent Foundry v2 (Prompt) agent
agent_name = "WriterAgent"

agent = client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You are a helpful agent that writes engaging book stories.",
    ),
)
print(f"Agent created: {agent.name} (version {agent.version})")

In [ ]:
# Prepare the user input — v2 agents use the OpenAI Responses API (no threads/messages)
user_input = "Write a short story about a robot who discovers music."

openai_client = client.get_openai_client()
print("OpenAI client ready.")

In [ ]:
# Stream the agent's response via the Responses API
print("--- Agent Response ---\n")

stream = openai_client.responses.create(
    stream=True,
    input=user_input,
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()


In [ ]:
# Cleanup: delete the agent (all versions) when done
client.agents.delete(agent_name=agent.name)
print("\nAgent deleted.")